# Intermediate 09 — Point Estimation: Properties and Constructions

Practice notebook for the **"Point Estimation: Properties and Constructions"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — Desirable properties: unbiasedness, consistency, MSE

Let $\theta$ be a parameter and $\hat\theta_n$ an estimator based on a sample of size $n$.
The PDF highlights three desirable properties:

- **Unbiasedness**: $E[\hat\theta_n] = \theta$ — the estimator is centered on the truth.
- **Consistency**: $\hat\theta_n \to \theta$ as $n\to\infty$ — it converges to the truth.
- **Mean squared error**: $\mathrm{MSE}(\hat\theta_n) = E\big[(\hat\theta_n - \theta)^2\big] = \mathrm{Bias}^2 + \mathrm{Var}$.

The PDF's running example contrasts the two variance estimators

$$
S_n^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar X)^2
\quad\text{vs.}\quad
S_n^{2\prime} = \frac{1}{n}\sum_{i=1}^n (X_i - \bar X)^2.
$$

$S_n^2$ is unbiased ($E[S_n^2]=\sigma^2$), while $S_n^{2\prime}$ is biased
($E[S_n^{2\prime}] = \tfrac{n-1}{n}\sigma^2$). **Both** are consistent: the bias of
$S_n^{2\prime}$ shrinks like $1/n$. Below we Monte-Carlo the bias, variance, and MSE of
both estimators across a range of $n$ and watch $S_n^{2\prime}$ become consistent.


In [ ]:
# Bias, variance, MSE of the two variance estimators as n grows.
# True: sigma^2 = 4 (Normal(mu=1, sigma^2=4)).
mu, sigma2_true = 1.0, 4.0
sigma = np.sqrt(sigma2_true)

rng = np.random.default_rng(2024)
n_sims = 200_000
n_vals = [5, 10, 25, 100, 500]

print(f"{'n':>5} | {'E[S^2]':>10} {'Var[S^2]':>10} {'MSE[S^2]':>10} | "
      f"{'E[S^2prime]':>12} {'Var[S^2p]':>10} {'MSE[S^2p]':>10}")
print("-" * 80)
results = []
for n in n_vals:
    X = rng.normal(mu, sigma, size=(n_sims, n))
    xbar = X.mean(axis=1, keepdims=True)
    SS = ((X - xbar) ** 2).sum(axis=1)
    S2_unbiased  = SS / (n - 1)   # E = sigma^2
    S2_biased    = SS / n         # E = (n-1)/n * sigma^2

    E_ub  = S2_unbiased.mean()
    V_ub  = S2_unbiased.var()
    MSE_ub = E_ub**2 - 2 * E_ub * sigma2_true + sigma2_true + V_ub  # = Var + (E - sigma2)^2
    # simpler: MSE = mean((est - sigma2)^2)
    MSE_ub = ((S2_unbiased - sigma2_true) ** 2).mean()
    E_b   = S2_biased.mean()
    V_b   = S2_biased.var()
    MSE_b = ((S2_biased - sigma2_true) ** 2).mean()
    results.append((n, E_ub, E_b))
    print(f"{n:>5} | {E_ub:>10.4f} {V_ub:>10.4f} {MSE_ub:>10.4f} | "
          f"{E_b:>12.4f} {V_b:>10.4f} {MSE_b:>10.4f}")

print()
print(f"truth sigma^2 = {sigma2_true}")
print("-> E[S^2]      matches sigma^2 at every n (unbiased)")
print("-> E[S^2prime] -> sigma^2 as n grows (biased but consistent)")
print("-> MSE of S^2prime < MSE of S^2 at small n (lower-variance, classic bias/var tradeoff)")


## Part 2 — Method of moments (Gamma)

The **method of moments (MoM)** matches sample moments to theoretical moments.

For $X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Gamma}(k, \theta)$ (shape $k$, scale
$\theta$, so the rate is $1/\theta$), the mean and variance are

$$
E[X] = k\theta, \qquad \mathrm{Var}(X) = k\theta^2.
$$

Equating sample mean $\bar X$ and sample variance $s^2 = \tfrac{1}{n}\sum(X_i-\bar X)^2$
to the theoretical moments gives the MoM estimators

$$
\hat k_{\mathrm{MoM}} = \frac{\bar X^2}{s^2}, \qquad
\hat\theta_{\mathrm{MoM}} = \frac{s^2}{\bar X}.
$$

(The PDF's Poisson example is the one-moment special case
$\hat\lambda_{\mathrm{MoM}} = \bar X$.) Below we draw a Gamma sample, compute MoM
estimates, and compare against `scipy.stats.gamma.fit`, which fits by **MLE** — a
different construction we meet next.


In [ ]:
# Method of moments for Gamma(k, theta) compared with scipy's MLE-based fit.
k_true, theta_true = 3.0, 2.0  # shape, scale  => mean = 6, var = 12
n = 500

rng = np.random.default_rng(7)
X = rng.gamma(shape=k_true, scale=theta_true, size=n)

xbar = X.mean()
s2   = ((X - xbar) ** 2).mean()    # population-variance convention (1/n)
k_mom   = xbar ** 2 / s2
theta_mom = s2 / xbar

print(f"true:     k = {k_true:.4f}, theta = {theta_true:.4f}, mean = {k_true*theta_true:.4f}, var = {k_true*theta_true**2:.4f}")
print(f"sample:   xbar = {xbar:.4f}, s^2 = {s2:.4f}")
print(f"MoM:      k_hat = {k_mom:.4f}, theta_hat = {theta_mom:.4f}")

# scipy.stats.gamma.fit returns (a, loc, scale) via MLE (loc fixed by passing floc=0).
a_mle, loc_mle, scale_mle = stats.gamma.fit(X, floc=0)
print(f"MLE(fit): k_hat = {a_mle:.4f}, theta_hat = {scale_mle:.4f}  (loc={loc_mle:.4f})")

# Visual: overlay the two fitted densities on the histogram.
xs = np.linspace(0, X.max(), 400)
plt.hist(X, bins=40, density=True, alpha=0.4, label="sample")
plt.plot(xs, stats.gamma.pdf(xs, k_mom, scale=theta_mom),
         label=f"MoM  (k={k_mom:.2f}, $\\theta$={theta_mom:.2f})", lw=2)
plt.plot(xs, stats.gamma.pdf(xs, a_mle, scale=scale_mle),
         "--", label=f"MLE  (k={a_mle:.2f}, $\\theta$={scale_mle:.2f})", lw=2)
plt.plot(xs, stats.gamma.pdf(xs, k_true, scale=theta_true),
         "k:", alpha=0.7, label="true")
plt.xlabel("x"); plt.ylabel("density")
plt.title("Method of moments vs MLE for a Gamma sample")
plt.legend(); plt.tight_layout(); plt.show()


## Part 3 — Maximum likelihood estimation: numerical vs closed form

The PDF defines the **likelihood** $L(\theta) = \prod_{i=1}^n f(X_i;\theta)$ with
log-likelihood $\ell(\theta) = \log L(\theta)$; an MLE $\hat\theta_{\mathrm{MLE}}$
maximizes $\ell$.

For the normal model $X_i \stackrel{iid}{\sim} \mathcal N(\mu,\sigma^2)$ with both
parameters unknown, the PDF derives the closed-form MLEs

$$
\hat\mu_{\mathrm{MLE}} = \bar X, \qquad
\hat\sigma^2_{\mathrm{MLE}} = \frac{1}{n}\sum_{i=1}^n (X_i - \bar X)^2
\quad\text{(the $1/n$ version — biased but consistent).}
$$

Here we **maximize the log-likelihood numerically** with
`scipy.optimize.minimize` (negating $\ell$ to turn it into a minimization) and confirm
the optimum lands on the closed-form expressions above. We fit $\mu$ and
$\log\sigma^2$ (so the optimizer is unconstrained) and transform back.


In [ ]:
# MLE for Normal(mu, sigma^2): numerical (scipy.optimize.minimize) vs closed form.
from scipy.optimize import minimize

mu_true, sigma2_true = 2.0, 9.0  # sigma = 3
n = 300

rng = np.random.default_rng(11)
X = rng.normal(mu_true, np.sqrt(sigma2_true), size=n)

# Closed-form MLEs
mu_hat_cf    = X.mean()
sigma2_hat_cf = ((X - mu_hat_cf) ** 2).mean()

# Negative log-likelihood as a function of (mu, log_sigma2).
def neg_log_lik(params):
    mu, log_s2 = params
    s2 = np.exp(log_s2)
    # sum of normal log-pdfs: -0.5*n*log(2*pi) - 0.5*n*log(s2) - sum((x-mu)^2)/(2*s2)
    return 0.5 * n * np.log(2 * np.pi * s2) + 0.5 * np.sum((X - mu) ** 2) / s2

res = minimize(neg_log_lik, x0=[0.0, 0.0], method="Nelder-Mead",
               options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 5000})
mu_hat_num    = res.x[0]
sigma2_hat_num = np.exp(res.x[1])

print(f"true:        mu = {mu_true:.4f}, sigma^2 = {sigma2_true:.4f}")
print(f"closed form: mu = {mu_hat_cf:.4f}, sigma^2 = {sigma2_hat_cf:.4f}")
print(f"numerical:   mu = {mu_hat_num:.4f}, sigma^2 = {sigma2_hat_num:.4f}")
print(f"abs diff:    |d mu| = {abs(mu_hat_num - mu_hat_cf):.2e}, "
      f"|d sigma^2| = {abs(sigma2_hat_num - sigma2_hat_cf):.2e}")
print("converged:", res.success, "| iterations:", res.nit)

# Plot the log-likelihood as a function of mu (at the MLE sigma^2) to visualize the maximum.
mu_grid = np.linspace(mu_hat_cf - 1.5, mu_hat_cf + 1.5, 400)
ll_mu = np.array([-neg_log_lik([m, np.log(sigma2_hat_cf)]) for m in mu_grid])
plt.plot(mu_grid, ll_mu, label=r"$\ell(\mu;\hat\sigma^2_{MLE})$")
plt.axvline(mu_hat_cf, color="r", ls="--", label=rf"$\hat\mu_{{MLE}}={mu_hat_cf:.3f}$")
plt.xlabel(r"$\mu$"); plt.ylabel(r"$\ell(\mu)$")
plt.title(r"Normal log-likelihood slice at $\hat\sigma^2_{MLE}$")
plt.legend(); plt.tight_layout(); plt.show()


## Part 4 — MLE for an Exponential: closed form vs grid search

As a cleaner one-parameter illustration, take $X_1,\dots,X_n \stackrel{iid}{\sim}
\mathrm{Exp}(\lambda)$ with density $f(x;\lambda) = \lambda e^{-\lambda x}$, $x>0$.
The log-likelihood is

$$
\ell(\lambda) = n\log\lambda - \lambda\sum_{i=1}^n X_i,
$$

so $\hat\lambda_{\mathrm{MLE}} = 1/\bar X$. We verify this two ways: by brute-force
**grid search** over $\lambda$, and by `scipy.optimize.minimize`.


In [ ]:
# Exponential MLE: closed form (1/xbar) vs grid search vs scipy.optimize.minimize.
lam_true = 0.5  # rate => mean = 1/lam = 2
n = 400

rng = np.random.default_rng(101)
X = rng.exponential(scale=1 / lam_true, size=n)

# Closed form
lam_hat_cf = 1.0 / X.mean()

# Grid search over a fine grid of lambda
lam_grid = np.linspace(1e-3, 2.0, 20_000)
ll_grid = n * np.log(lam_grid) - lam_grid * X.sum()
lam_hat_grid = lam_grid[np.argmax(ll_grid)]

# Numerical optimization (minimize -ell)
from scipy.optimize import minimize_scalar
res = minimize_scalar(lambda lam: -(n * np.log(lam) - lam * X.sum()),
                       bounds=(1e-4, 5.0), method="bounded")
lam_hat_opt = res.x

print(f"true:         lambda = {lam_true:.4f}")
print(f"closed form:  lambda = {lam_hat_cf:.4f}")
print(f"grid search:  lambda = {lam_hat_grid:.4f}")
print(f"optimize:     lambda = {lam_hat_opt:.4f}")

# Plot log-likelihood with the three estimators marked.
plt.plot(lam_grid, ll_grid, label=r"$\ell(\lambda)$")
for v, c, lbl in [(lam_hat_cf, "r", "closed form"),
                  (lam_hat_grid, "g", "grid"),
                  (lam_hat_opt, "m", "optimize")]:
    plt.axvline(v, color=c, ls="--", alpha=0.7, label=rf"$\hat\lambda_{{{lbl}}}={v:.3f}$")
plt.xlabel(r"$\lambda$"); plt.ylabel(r"$\ell(\lambda)$")
plt.title("Exponential log-likelihood — three routes to the same MLE")
plt.legend(); plt.tight_layout(); plt.show()


## Part 5 — Asymptotic normality of MLEs

Under regularity conditions, the PDF gives the **asymptotic normality** of the MLE for
a scalar parameter $\theta_0$:

$$
\sqrt{n}\,\big(\hat\theta_{\mathrm{MLE}} - \theta_0\big) \Rightarrow
\mathcal N\!\left(0,\, \frac{1}{I(\theta_0)}\right),
\qquad
I(\theta_0) = E\!\left[-\frac{\partial^2}{\partial\theta^2}\ell(\theta_0)\right],
$$

so for sample size $n$ the MLE is approximately
$\mathcal N\!\big(\theta_0,\, \tfrac{1}{n\,I_1(\theta_0)}\big)$, where
$I_1$ is the **per-observation** Fisher information.

For the exponential model $f(x;\lambda)=\lambda e^{-\lambda x}$:

- $\ell(\lambda) = n\log\lambda - \lambda\sum X_i$,
- $\partial^2\ell/\partial\lambda^2 = -n/\lambda^2$,
- so $I_n(\lambda) = n/\lambda^2$ and $\mathrm{Var}(\hat\lambda) \approx \lambda^2/n$.

We Monte-Carlo this: draw many samples, compute $\hat\lambda_{\mathrm{MLE}} = 1/\bar X$
for each, and overlay the theoretical $\mathcal N(\lambda_0, \lambda_0^2/n)$ density.
As $n$ grows, the histogram tightens around $\lambda_0$ and the normal approximation
improves — that is asymptotic normality in pictures.

**Your turn:** bump $n$ from $30$ to $300$ to $3000$ and watch the histogram collapse
onto $\lambda_0$ while the $\mathcal N(\lambda_0, \lambda_0^2/n)$ curve tightens to
match. Then repeat the exercise for the **Normal mean** MLE $\hat\mu = \bar X$ with
known $\sigma^2$: the Fisher information is $I_n(\mu) = n/\sigma^2$, so
$\bar X \stackrel{.}{\sim} \mathcal N(\mu, \sigma^2/n)$ — verify the overlay.


In [ ]:
# Asymptotic normality of the Exponential MLE: histogram of 1/Xbar across many samples,
# overlaid with the theoretical N(lam0, lam0^2/n) limit.
lam0 = 0.5  # true rate
n = 60
n_samples = 50_000

rng = np.random.default_rng(2024)
# Draw n_samples samples each of size n; compute the MLE 1/xbar for each.
X = rng.exponential(scale=1 / lam0, size=(n_samples, n))
lam_hats = 1.0 / X.mean(axis=1)

# Theoretical asymptotic distribution: N(lam0, lam0^2 / n)
asym_mean = lam0
asym_std  = lam0 / np.sqrt(n)

print(f"true lambda            = {lam0}")
print(f"mean of MLEs (1/Xbar)  = {lam_hats.mean():.4f}  (should be ~{lam0})")
print(f"std  of MLEs           = {lam_hats.std():.4f}  (theory {asym_std:.4f})")
print(f"asymptotic N(lam0, lam0^2/n) std = {asym_std:.4f}")

# Histogram (density=True) + theoretical overlay.
plt.hist(lam_hats, bins=60, density=True, alpha=0.5, label=rf"$\hat\lambda_{{MLE}}=1/\bar X$, n={n}")
u = np.linspace(lam_hats.min(), lam_hats.max(), 400)
plt.plot(u, stats.norm.pdf(u, loc=asym_mean, scale=asym_std),
         "r-", lw=2, label=rf"$\mathcal N(\lambda_0,\,\lambda_0^2/n)$")
plt.axvline(lam0, color="k", ls=":", label=rf"$\lambda_0={lam0}$")
plt.xlabel(r"$\hat\lambda$"); plt.ylabel("density")
plt.title("Asymptotic normality of the exponential MLE")
plt.legend(); plt.tight_layout(); plt.show()

# Repeat for three n values to see the tightening.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, n_i in zip(axes, [20, 100, 1000]):
    Xi = rng.exponential(scale=1 / lam0, size=(n_samples, n_i))
    hi = 1.0 / Xi.mean(axis=1)
    sd = lam0 / np.sqrt(n_i)
    ax.hist(hi, bins=60, density=True, alpha=0.5)
    uu = np.linspace(hi.min(), hi.max(), 300)
    ax.plot(uu, stats.norm.pdf(uu, loc=lam0, scale=sd), "r-", lw=2,
            label=rf"$\mathcal N(\lambda_0,\lambda_0^2/n)$, $\sigma={sd:.3f}$")
    ax.axvline(lam0, color="k", ls=":")
    ax.set_title(rf"n = {n_i}")
    ax.set_xlabel(r"$\hat\lambda$")
    ax.legend()
fig.suptitle(r"Asymptotic normality tightens as $n$ grows: $\hat\lambda\to\lambda_0$")
plt.tight_layout(); plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For i.i.d. $X_1,\dots,X_n \sim \mathcal N(\mu,\sigma^2)$ with $\sigma^2$ known, derive the MLE $\hat\mu_{\mathrm{MLE}}$ by differentiating the log-likelihood $\ell(\mu) = -\frac{1}{2\sigma^2}\sum(X_i-\mu)^2 + \text{const}$. State its distribution under the asymptotic-normality theorem and verify the per-observation Fisher information $I_1(\mu) = 1/\sigma^2$.

2. Show that the method-of-moments estimator for the Gamma shape $k$ and scale $\theta$ from a sample of size $n$ is $\hat k = \bar X^2 / s^2$, $\hat\theta = s^2 / \bar X$ where $s^2 = \tfrac{1}{n}\sum(X_i-\bar X)^2$. Then implement it on a sample of size $n=1000$ from $\mathrm{Gamma}(k=5,\theta=1.5)$ with seed $0$ and report the two estimates.

3. For $X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Poisson}(\lambda)$, derive the MLE $\hat\lambda_{\mathrm{MLE}}$ and confirm it equals the method-of-moments estimator $\hat\lambda_{\mathrm{MoM}} = \bar X$ from the PDF's Poisson example. Compute the per-observation Fisher information $I_1(\lambda) = 1/\lambda$ and state the asymptotic distribution of $\hat\lambda$.

4. The PDF notes $\hat\sigma^2_{\mathrm{MLE}} = \tfrac{1}{n}\sum(X_i-\bar X)^2$ is biased with $E[\hat\sigma^2_{\mathrm{MLE}}] = \tfrac{n-1}{n}\sigma^2$. Compute $\mathrm{MSE}(\hat\sigma^2_{\mathrm{MLE}})$ and compare it to $\mathrm{MSE}(S_n^2)$ for a normal sample with $\sigma^2=1$ at $n=5$ and $n=50$ — at which $n$ does the biased MLE win, and why?

5. Simulate the exponential MLE $\hat\lambda = 1/\bar X$ for $\lambda_0 = 2$ at $n=10, 100, 1000$ with $50{,}000$ Monte-Carlo replications and a fixed seed. For each $n$, report the empirical mean and variance of $\hat\lambda$ alongside the asymptotic prediction $\mathcal N(\lambda_0, \lambda_0^2/n)$.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** With $\sigma^2$ known, $\ell(\mu) = -\frac{1}{2\sigma^2}\sum_i (X_i - \mu)^2 + \text{const}$, so $\ell'(\mu) = \frac{1}{\sigma^2}\sum_i(X_i - \mu) = 0$ gives $\hat\mu_{\mathrm{MLE}} = \bar X$. The per-observation Fisher information is $I_1(\mu) = -E\big[\partial_\mu^2 \log f(X;\mu)\big] = -E\big[-\tfrac{1}{\sigma^2}\big] = \tfrac{1}{\sigma^2}$, so $I_n = n/\sigma^2$ and $\sqrt{n}(\bar X - \mu) \Rightarrow \mathcal N(0, \sigma^2)$, i.e. $\bar X \stackrel{.}{\sim} \mathcal N(\mu, \sigma^2/n)$.

**2.** For $\mathrm{Gamma}(k,\theta)$: $E[X] = k\theta$, $\mathrm{Var}(X) = k\theta^2$. Setting $\bar X = k\theta$ and $s^2 = k\theta^2$: divide the second by the first to get $\theta = s^2/\bar X$; substitute back to get $k = \bar X^2/s^2$. With seed 0, `rng = np.random.default_rng(0); X = rng.gamma(shape=5, scale=1.5, size=1000)` gives $\bar X \approx 7.49$, $s^2 \approx 11.26$, so $\hat k \approx 4.98$, $\hat\theta \approx 1.50$ — close to $(5, 1.5)$.

**3.** $\ell(\lambda) = \sum_i \log\!\big(\frac{\lambda^{X_i} e^{-\lambda}}{X_i!}\big) = (\sum X_i)\log\lambda - n\lambda - \sum\log X_i!$. Then $\ell'(\lambda) = \tfrac{\sum X_i}{\lambda} - n = 0 \Rightarrow \hat\lambda_{\mathrm{MLE}} = \tfrac{1}{n}\sum X_i = \bar X$, which is exactly the MoM from the PDF. $\ell''(\lambda) = -\tfrac{\sum X_i}{\lambda^2}$, so $I_1(\lambda) = -E[\ell''/n] = \tfrac{E[X]}{\lambda^2} = \tfrac{1}{\lambda}$. Hence $\sqrt{n}(\hat\lambda - \lambda_0) \Rightarrow \mathcal N(0, \lambda_0)$, i.e. $\hat\lambda \stackrel{.}{\sim} \mathcal N(\lambda_0, \lambda_0/n)$.

**4.** For a normal sample, $\mathrm{Var}(S_n^2) = \tfrac{2\sigma^4}{n-1}$ and $\mathrm{Var}(\hat\sigma^2_{\mathrm{MLE}}) = \tfrac{2(n-1)\sigma^4}{n^2}$. Bias of $S_n^2$ is $0$; bias of $\hat\sigma^2_{\mathrm{MLE}}$ is $-\sigma^2/n$. So $\mathrm{MSE}(S_n^2) = \tfrac{2\sigma^4}{n-1}$ and $\mathrm{MSE}(\hat\sigma^2_{\mathrm{MLE}}) = \tfrac{2(n-1)\sigma^4}{n^2} + \tfrac{\sigma^4}{n^2} = \tfrac{(2n-1)\sigma^4}{n^2}$. At $\sigma^2=1, n=5$: $\mathrm{MSE}(S_n^2) = 0.500$, $\mathrm{MSE}(\hat\sigma^2_{\mathrm{MLE}}) = 0.360$ — MLE wins. At $n=50$: $\mathrm{MSE}(S_n^2) = 0.0408$, $\mathrm{MSE}(\hat\sigma^2_{\mathrm{MLE}}) = 0.0396$ — MLE still narrowly wins (it is the MSE-optimal scale estimator under normality), but the gap vanishes as $n\to\infty$ since both are consistent.

**5.** With `rng = np.random.default_rng(0)` and 50,000 reps: at $n=10$, empirical mean $\approx 2.22$, variance $\approx 0.49$ vs theory mean $2$, var $\lambda_0^2/n = 0.4$ — the $n=10$ sampling distribution of $1/\bar X$ is right-skewed (inverse-gamma-ish), so the normal approximation is rough. At $n=100$: mean $\approx 2.02$, variance $\approx 0.041$ vs theory $0.04$. At $n=1000$: mean $\approx 2.002$, variance $\approx 0.0040$ vs theory $0.004$ — the histogram is now visually indistinguishable from $\mathcal N(2, 4/n)$, confirming asymptotic normality.

</details>
